In [ ]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler
from qiskit_experiments.library import StateTomography
from qiskit.quantum_info import Statevector, state_fidelity
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit.circuit.library import SGate, XGate, UnitaryGate
from qiskit.quantum_info import Operator

np.random.seed(42)
theta = np.random.uniform(0.0, np.pi)
varphi = np.random.uniform(0.0, 2*np.pi)

# 1. Provide your IBM Cloud API key and Instance CRN
service = QiskitRuntimeService(channel="ibm_cloud",
            token="", # Use the 44-character API_KEY you created and saved from the IBM Quantum Platform Home dashboard
            instance='') 
service.backends()

# 2. Select the backend
backend = service.least_busy(min_num_qubits=5, operational=True, simulator=False)

print(f"Running on: {backend.name}")

def run_teleportation_hardware(input_state_vector):
    # Create Circuit
    qc = QuantumCircuit(3, 2)
    qc.u(theta, varphi, 0.0, secret)
    qc.barrier()
    
    # Bell pair (Q1 and Q2)
    qc.h(1)
    qc.cx(1, 2)
    qc.barrier()
    
    # Alice's gates
    qc.cx(0, 1)
    qc.h(0)
    
    # Mid-circuit measurements
    qc.measure(0, 0)
    qc.measure(1, 1)
    qc.barrier()
    
    # Bob's corrections (Dynamic Circuits)
    # Using explicit indices for hardware stability
    with qc.if_test((qc.clbits[1], 1)):
        qc.x(2)
    with qc.if_test((qc.clbits[0], 1)):
        qc.z(2)
    
    # 2. Setup State Tomography on Bob's qubit (index 2)
    tomo_exp = StateTomography(qc, physical_qubits=(0, 1, 2), measurement_indices=[2])
    
    # 3. SET TRANSPILE OPTIONS (FIXED)
    tomo_exp.set_transpile_options(optimization_level=3)

    # 4. Execute using the Runtime Sampler V2
    sampler = Sampler(mode=backend)
    tomo_data = tomo_exp.run(backend, sampler=sampler, shots=4096).block_for_results()
    
    # 5. Results and Fidelity
    rho_fit = tomo_data.analysis_results("state").value
    target_state = Statevector(input_state_vector)
    fidelity = state_fidelity(target_state, rho_fit)
    
    return fidelity, rho_fit

# Execution
input_state = [1/np.sqrt(2), 1/np.sqrt(2)] # |+> state
fid, rho = run_teleportation_hardware(input_state)
print(f"Hardware Teleportation Fidelity: {fid:.5f}")


Running on: ibm_fez
Hardware Teleportation Fidelity: 0.94653


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_22696\3920614226.py:62: DeprecationWarning: Leaving `dataframe` unset or setting it to `False` for `ExperimentData.analysis_results` is deprecated as of qiskit-experiments 0.9.0. Future releases may change the default to `True` and remove the option to set the value to `False`.
  rho_fit = tomo_data.analysis_results("state").value


In [ ]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler
from qiskit_experiments.library import StateTomography
from qiskit.quantum_info import Statevector, state_fidelity

# 1. Initialize IBM Runtime Service
# Replace with your 44-character API token and instance (e.g., 'hub/group/project' or CRN)
import numpy as np
from qiskit.circuit.library import UnitaryGate
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler

service = QiskitRuntimeService(channel="ibm_cloud",
            token="", # Use the 44-character API_KEY you created and saved from the IBM Quantum Platform Home dashboard
            instance='') 
service.backends()


# Find the least busy real quantum processor (minimun 2 qubits)
backend = service.least_busy(operational=True, simulator=False, min_num_qubits=2)
print(f"Connected to backend: {backend.name}")

def run_sdc_hardware_tomography(pair, backend):
    # 2. Build the circuit (identical to your simulator version)
    qc = QuantumCircuit(2)
    qc.h(1)
    qc.cx(1, 0)
    qc.barrier()
    
    if pair[0] == '1': qc.z(1)
    if pair[1] == '1': qc.x(1)
    qc.barrier()
    
    qc.cx(1, 0)
    qc.h(1)

    # 3. Setup State Tomography
    # The ideal state Bob should receive is the basis state of the 'pair'
    target_state = Statevector.from_label(pair)
    qst_exp = StateTomography(qc)

    # 4. Execute on Hardware using Sampler V2
    # In 2026, Experiments require the sampler argument for runtime execution
    sampler = Sampler(mode=backend)
    
    # Run the experiment. Tomography will automatically transpile for the device.
    # Higher shots (e.g., 10000) are recommended for hardware to mitigate noise.
    tomo_data = qst_exp.run(backend, sampler=sampler, shots=10000).block_for_results()
    
    # 5. Extract results
    rho_fit = tomo_data.analysis_results("state").value
    fidelity = state_fidelity(target_state, rho_fit)
    
    return fidelity

# Main Execution Loop
for p in ['00', '01', '10', '11']:
    hw_fidelity = run_sdc_hardware_tomography(p, backend)
    print(f"Hardware Fidelity for message {p}: {hw_fidelity:.5f}")


Connected to backend: ibm_fez


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_28648\557210624.py:52: DeprecationWarning: Leaving `dataframe` unset or setting it to `False` for `ExperimentData.analysis_results` is deprecated as of qiskit-experiments 0.9.0. Future releases may change the default to `True` and remove the option to set the value to `False`.
  rho_fit = tomo_data.analysis_results("state").value


Hardware Fidelity for message 00: 0.88965


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_28648\557210624.py:52: DeprecationWarning: Leaving `dataframe` unset or setting it to `False` for `ExperimentData.analysis_results` is deprecated as of qiskit-experiments 0.9.0. Future releases may change the default to `True` and remove the option to set the value to `False`.
  rho_fit = tomo_data.analysis_results("state").value


Hardware Fidelity for message 01: 0.92212


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_28648\557210624.py:52: DeprecationWarning: Leaving `dataframe` unset or setting it to `False` for `ExperimentData.analysis_results` is deprecated as of qiskit-experiments 0.9.0. Future releases may change the default to `True` and remove the option to set the value to `False`.
  rho_fit = tomo_data.analysis_results("state").value


Hardware Fidelity for message 10: 0.89758
Hardware Fidelity for message 11: 0.92436


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_28648\557210624.py:52: DeprecationWarning: Leaving `dataframe` unset or setting it to `False` for `ExperimentData.analysis_results` is deprecated as of qiskit-experiments 0.9.0. Future releases may change the default to `True` and remove the option to set the value to `False`.
  rho_fit = tomo_data.analysis_results("state").value
